<a href="https://colab.research.google.com/github/Benxperia/MRes/blob/main/NDVI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the necessary libraries
!pip install rasterio numpy opencv-python

In [ ]:
# Import required libraries
import numpy as np
import cv2
import rasterio
import os
from google.colab import files

In [ ]:
# --- A. Upload the NDVI GeoTIFF ---
uploaded = files.upload()

if uploaded:
    uploaded_file_name = list(uploaded.keys())[0]
    print(f"\n✅ File uploaded successfully: {uploaded_file_name}")
else:
    uploaded_file_name = None
    print("❌ No file uploaded.")

# --- B. Define File Paths ---
INPUT_NDVI_RASTER = uploaded_file_name
OUTPUT_THRESHOLD_FILE = "/content/NDVI_Thresholds.txt"

Saving Chis_PlanetScope_NDVI.tif to Chis_PlanetScope_NDVI.tif

✅ File uploaded successfully: Chis_PlanetScope_NDVI.tif


In [ ]:
def calculate_ndvi_thresholds_colab(ndvi_path, output_threshold_path):
    """
    Calculates both the Otsu threshold (NDVI_soil) and the 98th percentile
    (NDVI_veg) from an NDVI raster.
    """

    # --- 1. Load the NDVI Raster using Rasterio ---
    import rasterio
    import numpy as np
    import cv2

    try:
        with rasterio.open(ndvi_path) as src:
            ndvi_array = src.read(1)
            nodata_value = src.nodata

    except Exception as e:
        print(f"An error occurred during file loading: {e}")
        return None, None

    # --- 2. Pre-process the Data ---
    if nodata_value is not None:
        ndvi_clean = ndvi_array[ndvi_array != nodata_value]

    ndvi_clean = ndvi_clean[~np.isnan(ndvi_clean)]
    ndvi_clean = ndvi_clean[(ndvi_clean >= -1) & (ndvi_clean <= 1)]

    if ndvi_clean.size == 0:
        print("Error: No valid data pixels remaining after filtering.")
        return None, None

    # --- 3. Scale and Convert for Otsu's Method ($NDVI_{soil}$) ---
    min_ndvi = -1.0
    max_ndvi = 1.0

    # Normalized array: (Value - Min) / (Max - Min)
    normalized_ndvi = (ndvi_clean - min_ndvi) / (max_ndvi - min_ndvi)

    # Scale to 0-255
    scaled_ndvi = (normalized_ndvi * 255).astype(np.uint8)

    # --- 4. Apply Otsu's Thresholding to find $NDVI_{soil}$ ---
    # ret_val is the threshold in the 0-255 range
    ret_val, _ = cv2.threshold(scaled_ndvi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Unscale the Otsu Threshold back to the original NDVI range (-1 to 1)
    # Unscaled = (Scaled / 255) * (Max - Min) + Min
    ndvi_soil_threshold = (ret_val / 255.0) * (max_ndvi - min_ndvi) + min_ndvi

    # --- 5. Calculate $NDVI_{veg}$ using the 98th Percentile ---
    # The 98th percentile is a robust estimate for the maximum NDVI value,
    # filtering out extreme noise/outliers.
    ndvi_veg_threshold = np.percentile(ndvi_clean, 98)

    # --- 6. Save and Report ---

    try:
        # Save both values to a text file
        with open(output_threshold_path, 'w') as f:
            f.write(f"NDVI_soil (Otsu Threshold): {ndvi_soil_threshold:.4f}\n")
            f.write(f"NDVI_veg (98th Percentile): {ndvi_veg_threshold:.4f}")

        print(f"\n✅ Threshold Calculations Complete.")
        print(f"   NDVI_soil (Bare Soil/Water) Candidate: {ndvi_soil_threshold:.4f}")
        print(f"   NDVI_veg (Dense Vegetation) Candidate: {ndvi_veg_threshold:.4f}")

    except IOError as e:
        print(f"\n❌ ERROR: Failed to save the output file to {output_threshold_path}")
        print(f"   Reason: {e}")
        return None, None

    return ndvi_soil_threshold, ndvi_veg_threshold

In [ ]:
if INPUT_NDVI_RASTER and os.path.exists(INPUT_NDVI_RASTER):
    # Run the calculation
    ndvi_soil_candidate, ndvi_veg_candidate = calculate_ndvi_thresholds_colab(
        INPUT_NDVI_RASTER,
        OUTPUT_THRESHOLD_FILE
    )

    if ndvi_soil_candidate is not None:
        print("\n--- Next Step: Fractional Vegetation Cover ($F_c$) ---")
        print(f"To calculate $F_c$, you will use the formula:")
        print(r"$$F_c = \frac{NDVI - NDVI_{soil}}{NDVI_{veg} - NDVI_{soil}}$$")
        print(f"Where:")
        print(f"  $NDVI_{{soil}} = {ndvi_soil_candidate:.4f}$")
        print(f"  $NDVI_{{veg}} = {ndvi_veg_candidate:.4f}$")

        # Download the results file
        files.download(OUTPUT_THRESHOLD_FILE)
        print(f"\n\u2705 **{os.path.basename(OUTPUT_THRESHOLD_FILE)}** has been downloaded to your computer.")
else:
    print("Cannot run calculation: Input file path is invalid or file was not uploaded.")


✅ Threshold Calculations Complete.
   NDVI_soil (Bare Soil/Water) Candidate: 0.3333
   NDVI_veg (Dense Vegetation) Candidate: 0.8780

--- Next Step: Fractional Vegetation Cover ($F_c$) ---
To calculate $F_c$, you will use the formula:
$$F_c = \frac{NDVI - NDVI_{soil}}{NDVI_{veg} - NDVI_{soil}}$$
Where:
  $NDVI_{soil} = 0.3333$
  $NDVI_{veg} = 0.8780$


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ **NDVI_Thresholds.txt** has been downloaded to your computer.
